# EMLProverV2 — Conjecture Engine Demo

This notebook demonstrates the new capabilities of `EMLProverV2`:
- **Conjecture generation**: grammar-based mutation of catalog identities
- **Proof visualization**: matplotlib tree diagrams of EML witnesses
- **Proof compression**: finding minimal EML witness trees
- **Batch proving**: mass-evaluation on the 120+ identity catalog


In [ ]:
from monogate.prover import EMLProverV2
from monogate.identities import ALL_IDENTITIES, get_by_category, get_by_difficulty

p = EMLProverV2(n_probe=500, verbose=False)
print(f"EMLProverV2 ready. Catalog size: {len(ALL_IDENTITIES)} identities.")

## 1. Identity Catalog Overview

In [ ]:
from collections import Counter

by_cat = Counter(i.category for i in ALL_IDENTITIES)
by_diff = Counter(i.difficulty for i in ALL_IDENTITIES)

print("By category:")
for cat, count in sorted(by_cat.items()):
    print(f"  {cat:20s} {count:3d}")

print("\nBy difficulty:")
for diff, count in sorted(by_diff.items()):
    print(f"  {diff:10s} {count:3d}")

## 2. Conjecture Generation

In [ ]:
# Generate novel trigonometric conjectures by grammar mutation
conjectures = p.generate_conjectures(category="trig", n=8, seed=42)

print(f"Generated {len(conjectures)} novel trig conjectures:\n")
for i, c in enumerate(conjectures):
    print(f"  [{i+1}] {c.name}")
    print(f"       {c.expression}")
    print(f"       difficulty={c.difficulty}, domain={c.domain}")
    print()

In [ ]:
# Prove the generated conjectures
print("Proving generated conjectures:\n")
for c in conjectures:
    r = p.prove(c.expression, domain=c.domain, n_simulations=200)
    status = "PROVED" if r.proved() else "FAILED"
    print(f"  [{status}] {c.expression[:60]}")

In [ ]:
# Generate from hyperbolic category
hyp_conjectures = p.generate_conjectures(category="hyperbolic", n=5, seed=7)
print(f"Generated {len(hyp_conjectures)} hyperbolic conjectures:")
for c in hyp_conjectures:
    print(f"  {c.expression}")

## 3. Live Proof Search

In [ ]:
import time

identities_to_prove = [
    "sin(x)**2 + cos(x)**2 == 1",
    "cosh(x)**2 - sinh(x)**2 == 1",
    "exp(x) * exp(-x) == 1",
    "log(exp(x)) == x",
]

print("Proving identities tier-by-tier:\n")
for expr in identities_to_prove:
    t0 = time.time()
    r = p.prove(expr, n_simulations=500)
    elapsed = time.time() - t0
    status = "✓" if r.proved() else "✗"
    print(f"{status} [{r.status:25s}] {expr[:50]:<50s}  ({elapsed:.3f}s)")

## 4. Proof Visualization

In [ ]:
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    result = p.prove("sin(x)**2 + cos(x)**2 == 1", n_simulations=500)
    print(f"Proof status: {result.status}")
    print(f"Witness formula: {result.witness_formula}")
    print(f"Node count: {result.node_count}")

    # Save tree visualization
    p.visualize_proof(result, style="tree", output_path="/tmp/proof_tree.png")
    print("Tree visualization saved to /tmp/proof_tree.png")

    # Step style
    p.visualize_proof(result, style="step", output_path="/tmp/proof_step.png")
    print("Step visualization saved to /tmp/proof_step.png")

except ImportError:
    print("matplotlib not available — install with: pip install matplotlib")

## 5. Proof Compression

In [ ]:
# Prove an identity that may have a redundant witness tree
result = p.prove("exp(x) - log(1) == exp(x)", n_simulations=1000)

print(f"Original proof:")
print(f"  Status:  {result.status}")
print(f"  Nodes:   {result.node_count}")
print(f"  Formula: {result.witness_formula}")

# Compress: find minimal equivalent tree
compressed = p.compress_proof(result, n_simulations=200)

print(f"\nCompressed proof:")
print(f"  Status:  {compressed.status}")
print(f"  Nodes:   {compressed.node_count}")
print(f"  Formula: {compressed.witness_formula}")
print(f"  Savings: {result.node_count - compressed.node_count} nodes")

## 6. Batch Proving — Catalog Benchmark

In [ ]:
# Prove all trivial and easy identities
easy_ids = [i for i in ALL_IDENTITIES if i.difficulty in ("trivial", "easy")][:20]

print(f"Batch proving {len(easy_ids)} identities...")
report = p.batch_prove(easy_ids, show_progress=False)

print(f"\nBenchmark Report:")
print(f"  Total:       {report.n_total}")
print(f"  Proved:      {report.n_proved}")
print(f"  Success rate: {report.success_rate:.1%}")
print(f"  Avg time:    {report.avg_time_s:.3f}s")

In [ ]:
# Show results table
print(f"{'Identity':<50s} {'Status':<25s} {'Nodes':<6s}")
print("-" * 85)
for r in report.results:
    status_mark = "✓" if r.proved() else "✗"
    identity_short = r.identity_str[:48]
    print(f"{status_mark} {identity_short:<48s} {r.status:<25s} {r.node_count:<6d}")

## 7. Mind-Blowing Gallery

Five remarkable identities from the catalog:

In [ ]:
gallery = [
    ("Pythagorean identity", "sin(x)**2 + cos(x)**2 == 1"),
    ("Euler's reflection (soft)", "exp(log(x)) == x"),
    ("Hyperbolic Pythagorean", "cosh(x)**2 - sinh(x)**2 == 1"),
    ("Double angle sin", "sin(2*x) == 2*sin(x)*cos(x)"),
    ("Log product rule", "log(exp(x) * exp(y)) == x + y"),
]

print("=" * 70)
print("EML MIND-BLOWING GALLERY")
print("=" * 70)

for name, expr in gallery:
    r = p.prove(expr, n_simulations=300)
    status = "PROVED" if r.proved() else "UNPROVED"
    confidence = f"{r.confidence:.6f}" if hasattr(r, 'confidence') else "N/A"
    print(f"\n{name}")
    print(f"  Identity:   {expr}")
    print(f"  Status:     {status} via {r.status}")
    print(f"  Method:     {r.proof_method}")
    print(f"  Witness:    {r.witness_formula}")